In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
df = pd.read_csv('../data/sexist_political_data.csv')
df

,Unnamed: 0,uniqueID,trigger_event,text,label
0,0,poli_ukdata_24-0,political,She has NOTHING on Braverman. Nothing. \n\nSue...,1
1,1,poli_ukdata_24-1,political,@10DowningStreet @SuellaBraverman @ukhomeoffic...,0
2,2,poli_ukdata_24-2,political,@_TheGMan @benandjerrysUK @SuellaBraverman Goo...,0
3,3,poli_ukdata_24-3,political,@Jerry_grey2002 @drmikechao @TsarKastik But I ...,0
4,4,poli_ukdata_24-4,political,@10DowningStreet @SuellaBraverman @ukhomeoffic...,0
...,...,...,...,...,...
1268,1268,poli_ukdata_24-1268,political,I am told by a Home Office source that civil s...,0
1269,1269,poli_ukdata_24-1269,political,"So Suella Braverman, who wants to make cannabi...",0
1270,1270,poli_ukdata_24-1270,political,:eyes:Boris Johnson and Simon Case suppressed ...,0
1271,1271,poli_ukdata_24-1271,political,"I do not care if Liz Truss had an affair, I do...",0


In [3]:
df['text'].iloc[0]

"She has NOTHING on Braverman. Nothing. \n\nSuella Braverman is the most dangerous, evil person ever to occupy high office in the UK. The only saving grace is that she's so completely batshit, she'll likely destroy herself. https://t.co/4RfPC57LLG"

In [4]:
df3 = pd.read_csv('../data/final_df.csv')
df3

,Unnamed: 0,label_1,label_2,label_3,uniqueID,text
0,0,not sexist,not sexist,sexist,antisexm24_uk-0,The rise of Suella Braverman is one of the gen...
1,1,not sexist,not sexist,not sexist,antisexm24_uk-1,"General Election please! 81,000 Tory members v..."
2,2,not sexist,not sexist,not sexist,antisexm24_uk-10,If you think you are having a tough day spare ...
3,3,not sexist,not sexist,not sexist,antisexm24_uk-100,"Already it seems Liz Truss"" ""energy price free..."
4,4,not sexist,not sexist,not sexist,antisexm24_uk-101,@pritipatel This is not about the location of ...
...,...,...,...,...,...,...
505,505,not sexist,not sexist,not sexist,antisexm24_uk-95,:eyes:Boris Johnson and Simon Case suppressed ...
506,506,not sexist,not sexist,not sexist,antisexm24_uk-96,Dear @CommonsSpeaker were you awake at #PMQs t...
507,507,not sexist,not sexist,not sexist,antisexm24_uk-97,"I do not care if Liz Truss had an affair, I do..."
508,508,sexist,not sexist,not sexist,antisexm24_uk-98,Priti Patel and Liz Truss are in posts at leas...


In [ ]:
# Fuzzy match rows between df and df3 by text similarity >= 0.70 using rapidfuzz, and remove duplicates
from rapidfuzz import process, fuzz

rows = []
for idx, row in df.iterrows():
    result = process.extractOne(
        row['text'], df3['text'], scorer=fuzz.token_sort_ratio, score_cutoff=70
    )
    if result is not None:
        match, score, match_idx = result
        combined = row.to_dict()
        for col in df3.columns:
            if col not in ['text']:
                combined[f'df3_{col}'] = df3.iloc[match_idx][col]
        rows.append(combined)

matched_df = pd.DataFrame(rows)
matched_df = matched_df.drop_duplicates()
matched_df.head()

In [ ]:
matched_df = matched_df.drop(columns=['df3_Unnamed: 0', 'Unnamed: 0', 'df3_uniqueID', 'label'])
matched_df

,uniqueID,trigger_event,text,df3_label_1,df3_label_2,df3_label_3
0,poli_ukdata_24-125,political,🚨#BREAKING: Suella Braverman returns as #UK Ho...,not sexist,not sexist,not sexist
1,poli_ukdata_24-137,political,@10DowningStreet @SuellaBraverman @ukhomeoffic...,sexist,not sexist,not sexist
2,poli_ukdata_24-549,political,Good riddance @pritipatel \n\nIt's been an emb...,not sexist,not sexist,not sexist
3,poli_ukdata_24-619,political,@DCBMEP Appointing Suella Braverman as Home se...,not sexist,not sexist,not sexist
4,poli_ukdata_24-1107,political,The rise of Suella Braverman is one of the gen...,not sexist,not sexist,sexist
...,...,...,...,...,...,...
164,poli_ukdata_24-1268,political,I am told by a Home Office source that civil s...,not sexist,not sexist,not sexist
165,poli_ukdata_24-1269,political,"So Suella Braverman, who wants to make cannabi...",not sexist,not sexist,not sexist
166,poli_ukdata_24-1270,political,:eyes:Boris Johnson and Simon Case suppressed ...,not sexist,not sexist,not sexist
167,poli_ukdata_24-1271,political,"I do not care if Liz Truss had an affair, I do...",not sexist,not sexist,not sexist


In [ ]:
matched_df.iloc[0]

uniqueID                                         poli_ukdata_24-125
trigger_event                                             political
text              🚨#BREAKING: Suella Braverman returns as #UK Ho...
df3_label_1                                              not sexist
df3_label_2                                              not sexist
df3_label_3                                              not sexist
majority_label                                                 None
minority_label                                                 None
Name: 0, dtype: object

In [ ]:
def get_majority_minority(row):
    labels = [row['df3_label_1'], row['df3_label_2'], row['df3_label_3']]
    from collections import Counter
    counts = Counter(labels)
    majority = counts.most_common(1)[0][0]
    minority = counts.most_common()[-1][0] if len(counts) > 1 else majority
    return pd.Series({'majority_label': majority, 'minority_label': minority})

matched_df[['majority_label', 'minority_label']] = matched_df.apply(get_majority_minority, axis=1)

In [ ]:
matched_df = matched_df.drop(columns={'df3_label_1', 'df3_label_2', 'df3_label_3'})

In [ ]:
matched_df

In [79]:
matched_df.to_csv('../data/matched_df.csv', index=False)

In [73]:
df4 = pd.read_csv('../data/majority_data.csv')
df4

,uniqueID,text,Annotation_Chico 1,Annotation_task_10,Annotation_task_3,Annotation_task_4,Annotation_task_6,Annotations_Aditi 1,Annotations_Aditi 2,Annotations_Aditi 3,...,Annotations_Chico 1,Annotations_Susan 1,Annotations_Susan 2,Annotations_Susan 3,Task_Chico_1,Task_Susan_1,Task_Susan_1_annotation_1,Task_me_2,Task_me_2_annotation_1,majority_label
0,poli_ukdata_24-827,"""Believe me"" says @AngelaRayner, a politician....",NaN,not sexist,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist
1,poli_ukdata_24-430,"""Boris Johnson hits out at ‘misogyny’ directed...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist,NaN,NaN,not sexist
2,poli_ukdata_24-331,"""I can call spirits from the vasty deep"". \n""W...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist
3,poli_ukdata_24-440,"""Somebody sent me the screenshot of it on Twit...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist
4,poli_ukdata_24-469,"""Today I have mainly been pretending I didn't ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
852,poli_ukdata_24-726,Written question from @EmilyThornberry. Cabine...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,not sexist,NaN,NaN,NaN,NaN,NaN,not sexist
853,poli_ukdata_24-963,You say so much that's so true Angela. https:/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist,NaN,NaN,not sexist
854,poli_ukdata_24-391,https://t.co/MXrAM78vHs\n\nTerrors of the Eart...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist
855,poli_ukdata_24-480,|'Amhara security forces are responsible for a...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist


In [49]:
df4.columns

Index(['uniqueID', 'text', 'Annotation_Chico 1', 'Annotation_task_10',
       'Annotation_task_3', 'Annotation_task_4', 'Annotation_task_6',
       'Annotations_Aditi 1', 'Annotations_Aditi 2', 'Annotations_Aditi 3',
       'Annotations_Aditi 4', 'Annotations_Chico 1', 'Annotations_Susan 1',
       'Annotations_Susan 2', 'Annotations_Susan 3', 'Task_Chico_1',
       'Task_Susan_1', 'Task_Susan_1_annotation_1', 'Task_me_2',
       'Task_me_2_annotation_1', 'majority_label'],
      dtype='object')

In [45]:
df4['uniqueID'].isin(df['uniqueID']).sum()

857

In [52]:
# 1. define annotation columns properly
annotation_cols = [
    col for col in df4.columns
    if col not in ["uniqueID", "text", "majority_label"]
]

def get_majority_minority(row):
    votes = row[annotation_cols].dropna().values
    
    if len(votes) == 0:
        return pd.Series([None, None])
    
    vc = pd.Series(votes).value_counts()
    
    majority = vc.idxmax()
    
    # minority = least frequent observed label
    minority = vc.idxmin()
    
    return pd.Series([majority, minority])

df4[["majority_label", "minority_label"]] = df4.apply(get_majority_minority, axis=1)

In [54]:
import pandas as pd

def process_annotations(df):
    # 1. identify annotation columns
    annotation_cols = [
        col for col in df.columns
        if col not in ["uniqueID", "text", "majority_label", "minority_label"]
    ]

    # 2. keep only rows with at least 2 annotations
    df = df.copy()
    df["n_annotations"] = df[annotation_cols].notna().sum(axis=1)
    df = df[df["n_annotations"] > 1].copy()
    df.drop(columns=["n_annotations"], inplace=True)

    # 3. compute majority/minority labels
    def get_labels(row):
        votes = row[annotation_cols].dropna().values

        if len(votes) == 0:
            return pd.Series([None, None])

        vc = pd.Series(votes).value_counts()

        majority = vc.idxmax()

        other_labels = [l for l in vc.index if l != majority]
        minority = other_labels[0] if len(other_labels) > 0 else majority

        return pd.Series([majority, minority])

    df[["majority_label", "minority_label"]] = df.apply(get_labels, axis=1)

    return df

In [53]:
df4

,uniqueID,text,Annotation_Chico 1,Annotation_task_10,Annotation_task_3,Annotation_task_4,Annotation_task_6,Annotations_Aditi 1,Annotations_Aditi 2,Annotations_Aditi 3,...,Annotations_Susan 1,Annotations_Susan 2,Annotations_Susan 3,Task_Chico_1,Task_Susan_1,Task_Susan_1_annotation_1,Task_me_2,Task_me_2_annotation_1,majority_label,minority_label
0,poli_ukdata_24-827,"""Believe me"" says @AngelaRayner, a politician....",NaN,not sexist,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist
1,poli_ukdata_24-430,"""Boris Johnson hits out at ‘misogyny’ directed...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,not sexist,not sexist,NaN,NaN,not sexist,not sexist
2,poli_ukdata_24-331,"""I can call spirits from the vasty deep"". \n""W...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist
3,poli_ukdata_24-440,"""Somebody sent me the screenshot of it on Twit...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist
4,poli_ukdata_24-469,"""Today I have mainly been pretending I didn't ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
852,poli_ukdata_24-726,Written question from @EmilyThornberry. Cabine...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,not sexist,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist
853,poli_ukdata_24-963,You say so much that's so true Angela. https:/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,not sexist,not sexist,NaN,NaN,not sexist,not sexist
854,poli_ukdata_24-391,https://t.co/MXrAM78vHs\n\nTerrors of the Eart...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist
855,poli_ukdata_24-480,|'Amhara security forces are responsible for a...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist


In [55]:
processed_df4 = process_annotations(df4)
processed_df4

,uniqueID,text,Annotation_Chico 1,Annotation_task_10,Annotation_task_3,Annotation_task_4,Annotation_task_6,Annotations_Aditi 1,Annotations_Aditi 2,Annotations_Aditi 3,...,Annotations_Susan 1,Annotations_Susan 2,Annotations_Susan 3,Task_Chico_1,Task_Susan_1,Task_Susan_1_annotation_1,Task_me_2,Task_me_2_annotation_1,majority_label,minority_label
1,poli_ukdata_24-430,"""Boris Johnson hits out at ‘misogyny’ directed...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,not sexist,not sexist,NaN,NaN,not sexist,not sexist
29,poli_ukdata_24-617,@10DowningStreet @SuellaBraverman @ukhomeoffic...,not sexist,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist
55,poli_ukdata_24-1072,"@51Bee71 @baratheongirl @AngelaRayner Exactly,...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist,not sexist,not sexist
62,poli_ukdata_24-961,@AlStewartOBE @joerichlaw @AngelaRayner @Daily...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,not sexist,not sexist,NaN,NaN,not sexist,not sexist
63,poli_ukdata_24-952,@AlStewartOBE @stevebr98325879 @AngelaRayner @...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,not sexist,not sexist,NaN,NaN,not sexist,not sexist
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
841,poli_ukdata_24-1054,We're still on this flat issue when Brexit and...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist,not sexist,not sexist
843,poli_ukdata_24-1020,Well don’t look in the mirror. Simples. https:...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,sexist,sexist,sexist,sexist
851,poli_ukdata_24-1045,"Wow, there is Angela Rayner outside of Parliam...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,not sexist,not sexist,not sexist,not sexist
853,poli_ukdata_24-963,You say so much that's so true Angela. https:/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,not sexist,not sexist,NaN,NaN,not sexist,not sexist


In [56]:
(processed_df4['majority_label'] != processed_df4['minority_label']).sum()

3

In [57]:
(matched_df['majority_label'] != matched_df['minority_label']).sum()

12

In [8]:
df_alpaca = pd.read_csv('../data/llm_data_maj/alpaca.csv')
df_flan = pd.read_csv('../data/llm_data_maj/flan_t5.csv')
df_mistral = pd.read_csv('../data/llm_data_maj/mistral.csv')
df_vicuna = pd.read_csv('../data/llm_data_maj/vicuna.csv')

In [9]:
df_vicuna.head()

,Unnamed: 0,uniqueID,trigger_event,majority_label,minority_label,roleplay,content,zero-shot,few-shot
0,0,poli_ukdata_24-125,political,not sexist,not sexist,not sexist,sexist,not sexist,not sexist
1,1,poli_ukdata_24-137,political,not sexist,sexist,not sexist,sexist,not sexist,not sexist
2,2,poli_ukdata_24-549,political,not sexist,not sexist,not sexist,sexist,sexist,not sexist
3,3,poli_ukdata_24-619,political,not sexist,not sexist,not sexist,not sexist,sexist,not sexist
4,4,poli_ukdata_24-1107,political,not sexist,sexist,sexist,sexist,sexist,sexist


In [10]:
def compute_metrics(df, truth_col, pred_col):
    """
    Compute classification metrics for one prompt type.
    """

    y_true = df[truth_col]
    y_pred = df[pred_col]

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    }

In [11]:
def evaluate_model(
    df,
    truth_col,
    prompt_cols=["roleplay", "content", "zero-shot", "few-shot"]
):
    """
    Evaluate every prompt against a chosen ground truth.
    """

    results = []

    for prompt in prompt_cols:

        metrics = compute_metrics(
            df=df,
            truth_col=truth_col,
            pred_col=prompt
        )

        metrics["prompt"] = prompt

        results.append(metrics)

    results = pd.DataFrame(results)

    cols = [
        "prompt",
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1"
    ]

    return results[cols]

In [13]:
majority_results = evaluate_model(
    df_flan,
    truth_col="majority_label"
)

majority_results

,prompt,accuracy,macro_precision,macro_recall,macro_f1
0,roleplay,0.964497,0.580266,0.735030,0.615909
1,content,0.964497,0.493939,0.488024,0.490964
2,zero-shot,0.976331,0.494012,0.494012,0.494012
3,few-shot,0.970414,0.493976,0.491018,0.492492


In [14]:
def evaluate_all_models(
    model_dfs,
    truth_col,
    prompt_cols=["roleplay", "content", "zero-shot", "few-shot"]
):
    """
    model_dfs:
        {
            "flan": flan_df,
            "mistral": mistral_df,
            ...
        }
    """

    all_results = []

    for model_name, df in model_dfs.items():

        res = evaluate_model(
            df=df,
            truth_col=truth_col,
            prompt_cols=prompt_cols
        )

        res["model"] = model_name

        all_results.append(res)

    return pd.concat(all_results, ignore_index=True)

In [15]:
models = {
    "flan": df_flan,
    "mistral": df_mistral,
    "vicuna": df_vicuna,
    "alpaca": df_alpaca
}

In [16]:
majority_eval = evaluate_all_models(
    models,
    truth_col="majority_label"
)

majority_eval

,prompt,accuracy,macro_precision,macro_recall,macro_f1,model
0,roleplay,0.964497,0.580266,0.735030,0.615909,flan
1,content,0.964497,0.493939,0.488024,0.490964,flan
2,zero-shot,0.976331,0.494012,0.494012,0.494012,flan
3,few-shot,0.970414,0.493976,0.491018,0.492492,flan
4,roleplay,0.857988,0.493197,0.434132,0.461783,mistral
5,content,0.822485,0.492908,0.416168,0.451299,mistral
6,zero-shot,0.846154,0.493103,0.428144,0.458333,mistral
7,few-shot,0.952663,0.493865,0.482036,0.487879,mistral
8,roleplay,0.390533,0.497279,0.444611,0.288494,vicuna
9,content,0.556213,0.489583,0.281437,0.357414,vicuna


In [17]:
minority_eval = evaluate_all_models(
    models,
    truth_col="minority_label"
)

minority_eval

,prompt,accuracy,macro_precision,macro_recall,macro_f1,model
0,roleplay,0.905325,0.469325,0.481132,0.475155,flan
1,content,0.952663,0.853788,0.646855,0.701940,flan
2,zero-shot,0.940828,0.723054,0.546855,0.567996,flan
3,few-shot,0.946746,0.809237,0.596855,0.640000,flan
4,roleplay,0.846154,0.544372,0.590252,0.551266,mistral
5,content,0.798817,0.507345,0.518239,0.495965,mistral
6,zero-shot,0.810651,0.489799,0.477673,0.476780,mistral
7,few-shot,0.905325,0.469325,0.481132,0.475155,mistral
8,roleplay,0.390533,0.486393,0.441824,0.315359,vicuna
9,content,0.579882,0.520263,0.589308,0.433073,vicuna


In [18]:
def compute_prompt_deltas(
    metrics_df,
    baseline="roleplay"
):
    """
    Computes metric differences relative to baseline prompt.
    """

    out = []

    for model in metrics_df["model"].unique():

        model_df = metrics_df[
            metrics_df["model"] == model
        ]

        base = model_df[
            model_df["prompt"] == baseline
        ].iloc[0]

        for _, row in model_df.iterrows():

            result = row.to_dict()

            for metric in [
                "accuracy",
                "macro_precision",
                "macro_recall",
                "macro_f1"
            ]:

                result[f"delta_{metric}"] = (
                    row[metric] - base[metric]
                )

            out.append(result)

    return pd.DataFrame(out)

In [19]:
majority_deltas = compute_prompt_deltas(
    majority_eval,
    baseline="roleplay"
)

majority_deltas

,prompt,accuracy,macro_precision,macro_recall,macro_f1,model,delta_accuracy,delta_macro_precision,delta_macro_recall,delta_macro_f1
0,roleplay,0.964497,0.580266,0.735030,0.615909,flan,0.000000,0.000000,0.000000,0.000000
1,content,0.964497,0.493939,0.488024,0.490964,flan,0.000000,-0.086326,-0.247006,-0.124945
2,zero-shot,0.976331,0.494012,0.494012,0.494012,flan,0.011834,-0.086254,-0.241018,-0.121897
3,few-shot,0.970414,0.493976,0.491018,0.492492,flan,0.005917,-0.086290,-0.244012,-0.123417
4,roleplay,0.857988,0.493197,0.434132,0.461783,mistral,0.000000,0.000000,0.000000,0.000000
5,content,0.822485,0.492908,0.416168,0.451299,mistral,-0.035503,-0.000289,-0.017964,-0.010485
6,zero-shot,0.846154,0.493103,0.428144,0.458333,mistral,-0.011834,-0.000094,-0.005988,-0.003450
7,few-shot,0.952663,0.493865,0.482036,0.487879,mistral,0.094675,0.000668,0.047904,0.026095
8,roleplay,0.390533,0.497279,0.444611,0.288494,vicuna,0.000000,0.000000,0.000000,0.000000
9,content,0.556213,0.489583,0.281437,0.357414,vicuna,0.165680,-0.007695,-0.163174,0.068921


In [20]:
def make_f1_table(metrics_df):

    return metrics_df.pivot(
        index="model",
        columns="prompt",
        values="macro_f1"
    ).round(3)

In [21]:
make_f1_table(majority_eval)

prompt,content,few-shot,roleplay,zero-shot
model,,,,
alpaca,0.063,0.256,0.258,0.139
flan,0.491,0.492,0.616,0.494
mistral,0.451,0.488,0.462,0.458
vicuna,0.357,0.480,0.288,0.369


In [23]:
models = {
    "flan": df_flan,
    "mistral": df_mistral,
    "vicuna": df_vicuna,
    "alpaca": df_alpaca
}

majority_eval = evaluate_all_models(
    models,
    truth_col="majority_label"
)

minority_eval = evaluate_all_models(
    models,
    truth_col="minority_label"
)

majority_deltas = compute_prompt_deltas(
    majority_eval,
    baseline="roleplay"
)

minority_deltas = compute_prompt_deltas(
    minority_eval,
    baseline="roleplay"
)

display(make_f1_table(majority_eval))
display(make_f1_table(minority_eval))

prompt,content,few-shot,roleplay,zero-shot
model,,,,
alpaca,0.063,0.256,0.258,0.139
flan,0.491,0.492,0.616,0.494
mistral,0.451,0.488,0.462,0.458
vicuna,0.357,0.480,0.288,0.369


prompt,content,few-shot,roleplay,zero-shot
model,,,,
alpaca,0.101,0.332,0.279,0.184
flan,0.702,0.640,0.475,0.568
mistral,0.496,0.475,0.551,0.477
vicuna,0.433,0.568,0.315,0.450


In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(df, truth_col, pred_col):

    y_true = df[truth_col]
    y_pred = df[pred_col]

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }

In [25]:
def evaluate_models_all_truths(models, prompt_cols):
    
    rows = []

    for model_name, df in models.items():
        for prompt in prompt_cols:

            for truth in ["majority_label", "minority_label"]:

                metrics = compute_metrics(df, truth, prompt)

                metrics.update({
                    "model": model_name,
                    "prompt": prompt,
                    "truth_type": truth
                })

                rows.append(metrics)

    return pd.DataFrame(rows)

In [27]:
prompt_cols = ["roleplay", "content", "zero-shot", "few-shot"]

eval_df = evaluate_models_all_truths(models, prompt_cols)

eval_df

,accuracy,macro_precision,macro_recall,macro_f1,model,prompt,truth_type
0,0.964497,0.580266,0.735030,0.615909,flan,roleplay,majority_label
1,0.905325,0.469325,0.481132,0.475155,flan,roleplay,minority_label
2,0.964497,0.493939,0.488024,0.490964,flan,content,majority_label
3,0.952663,0.853788,0.646855,0.701940,flan,content,minority_label
4,0.976331,0.494012,0.494012,0.494012,flan,zero-shot,majority_label
5,0.940828,0.723054,0.546855,0.567996,flan,zero-shot,minority_label
6,0.970414,0.493976,0.491018,0.492492,flan,few-shot,majority_label
7,0.946746,0.809237,0.596855,0.640000,flan,few-shot,minority_label
8,0.857988,0.493197,0.434132,0.461783,mistral,roleplay,majority_label
9,0.846154,0.544372,0.590252,0.551266,mistral,roleplay,minority_label


In [28]:
def compute_majority_minority_deltas(eval_df):
    
    out = []

    grouped = eval_df.groupby(["model", "prompt"])

    for (model, prompt), group in grouped:

        maj = group[group["truth_type"] == "majority_label"].iloc[0]
        mino = group[group["truth_type"] == "minority_label"].iloc[0]

        row = {
            "model": model,
            "prompt": prompt
        }

        for metric in ["accuracy", "macro_precision", "macro_recall", "macro_f1"]:
            row[f"delta_{metric}"] = maj[metric] - mino[metric]

        out.append(row)

    return pd.DataFrame(out)

In [29]:
delta_df = compute_majority_minority_deltas(eval_df)

delta_df

,model,prompt,delta_accuracy,delta_macro_precision,delta_macro_recall,delta_macro_f1
0,alpaca,content,-0.035503,0.033681,0.051789,-0.037078
1,alpaca,few-shot,-0.047337,-0.036697,-0.408737,-0.076560
2,alpaca,roleplay,0.000000,0.020494,0.204595,-0.020893
3,alpaca,zero-shot,-0.035503,-0.003305,0.049529,-0.044488
4,flan,content,0.011834,-0.359848,-0.158831,-0.210976
5,flan,few-shot,0.023669,-0.315261,-0.105837,-0.147508
6,flan,roleplay,0.059172,0.110941,0.253898,0.140754
7,flan,zero-shot,0.035503,-0.229042,-0.052843,-0.073984
8,mistral,content,0.023669,-0.014438,-0.102071,-0.044666
9,mistral,few-shot,0.047337,0.024540,0.000904,0.012724


In [32]:
gemini_full_df = pd.read_csv('../data/llm_data/gemini.csv')
gemini_full_df

,Unnamed: 0.1,Unnamed: 0,uniqueID,trigger_event,text,label,roleplay_results,roleplay_vocab_probs,roleplay_ppl,content_results,content_vocab_probs,content_ppl,zeroshot_results,zeroshot_vocab_probs,zeroshot_ppl,fewshot_results,fewshot_vocab_probs,fewshot_ppl
0,0,0,poli_ukdata_24-0,political,She has NOTHING on Braverman. Nothing. \n\nSue...,1,"['not sexist', 'not sexist', 'not sexist']","[[0.04999991336767482, 0.05000004179888724, 0....","[1.000019374801589, 1.000019374801589, 1.00001...","['not sexist', 'not sexist', 'not sexist']","[[0.04999986241318358, 0.049999987907874036, 0...","[1.0000020594667773, 1.0000020594667773, 1.000...","['not sexist', 'not sexist', 'not sexist']","[[0.05000003767888987, 0.0500001107871465, 0.0...","[1.0000019489495493, 1.0000015761096421, 1.000...","['not sexist', 'not sexist', 'not sexist']","[[0.04999994390758528, 0.050000013388841004, 0...","[1.000106979758213, 1.000106979758213, 1.00010..."
1,1,1,poli_ukdata_24-1,political,@10DowningStreet @SuellaBraverman @ukhomeoffic...,0,"['sexist', 'sexist', 'sexist']","[[0.050002987092221764, 0.05000017339726531, 0...","[1.092626968218736, 1.092626968218736, 1.07864...","['not sexist', 'not sexist', 'not sexist']","[[0.05000004098905716, 0.05000004790420219, 0....","[0.9999997522682307, 0.9999997522682307, 0.999...","['not sexist', 'not sexist', 'not sexist']","[[0.04999985612054209, 0.05000004140685375, 0....","[1.0000058566391834, 1.0000030224567475, 1.000...","['not sexist', 'not sexist', 'not sexist']","[[0.049999877805903224, 0.050000102073318564, ...","[1.0000117747806556, 1.0000212318955934, 1.000..."
2,2,2,poli_ukdata_24-2,political,@_TheGMan @benandjerrysUK @SuellaBraverman Goo...,0,"['not sexist', 'not sexist', 'not sexist']","[[0.05000016098581337, 0.050000080865731336, 0...","[1.0000138490174635, 1.0000138490174635, 1.000...","['not sexist', 'not sexist', 'not sexist']","[[0.050000000345254925, 0.05000008861551699, 0...","[0.9999995900629417, 0.999998521060827, 0.9999...","['not sexist', 'not sexist', 'not sexist']","[[0.05000005344604051, 0.05000018023614846, 0....","[0.9999990781074583, 0.9999998044222524, 0.999...","['not sexist', 'not sexist', 'not sexist']","[[0.05000019075490011, 0.05000013550868776, 0....","[1.0000068672857463, 1.0000068672857463, 1.000..."
3,3,3,poli_ukdata_24-3,political,@Jerry_grey2002 @drmikechao @TsarKastik But I ...,0,"['not sexist', 'not sexist', 'not sexist']","[[0.05000004179919341, 0.05000013163173437, 0....","[1.000018153475239, 1.000018153475239, 1.00001...","['not sexist', 'not sexist', 'not sexist']","[[0.05000019592140881, 0.05000019141515518, 0....","[0.9999984561787251, 0.9999984561787251, 1.000...","['not sexist', 'not sexist', 'not sexist']","[[0.05000017596986345, 0.05000005396522104, 0....","[1.0000002760208047, 1.0000005662442704, 1.000...","['not sexist', 'not sexist', 'not sexist']","[[0.04999999990352032, 0.050000039070275795, 0...","[1.0000066953003182, 1.0000103945446897, 1.000..."
4,4,4,poli_ukdata_24-4,political,@10DowningStreet @SuellaBraverman @ukhomeoffic...,0,"['not sexist', 'not sexist', 'not sexist']","[[0.05000008399964913, 0.04999995212106706, 0....","[1.0000271082189878, 1.0000271082189878, 1.000...","['not sexist', 'not sexist', 'not sexist']","[[0.05000002289633019, 0.05000019115452574, 0....","[0.9999992098701221, 1.0000016158459055, 0.999...","['not sexist', 'not sexist', 'not sexist']","[[0.04999983185008471, 0.05000010126891717, 0....","[1.0000011523571306, 1.000001470578848, 1.0000...","['not sexist', 'not sexist', 'not sexist']","[[0.05000002462465488, 0.05000002980313452, 0....","[1.0000928345081064, 1.0000928345081064, 1.000..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1268,1268,1268,poli_ukdata_24-1268,political,I am told by a Home Office source that civil s...,0,"['not sexist', 'not sexist', 'not sexist']","[[0.05000003328444132, 0.04999992825027006, 0....","[1.0000192384378563, 1.0000116711019935, 1.000...","['not sexist', 'not sexist'